In [1]:
import os
import torch
import numpy as np
from PIL import Image
import torchvision.transforms as transforms
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import lpips
from transformers import CLIPProcessor, CLIPModel
import cv2
import json
from tqdm import tqdm

def normalize_psnr(psnr_value):
    """Normalize PSNR to 0-1 range"""
    min_psnr = 20
    max_psnr = 50
    return np.clip((psnr_value - min_psnr) / (max_psnr - min_psnr), 0, 1)

def calculate_combined_score(psnr_value, ssim_value, lpips_value, clip_sim):
    """Calculate combined similarity score with weighted metrics"""
    norm_psnr = normalize_psnr(psnr_value)
    norm_ssim = (ssim_value + 1) / 2
    norm_lpips = 1 - lpips_value
    norm_clip = (clip_sim + 1) / 2
    
    weights = {
        'psnr': 0.15,
        'ssim': 0.30,
        'lpips': 0.35,
        'clip': 0.20
    }
    
    combined_score = (
        weights['psnr'] * norm_psnr +
        weights['ssim'] * norm_ssim +
        weights['lpips'] * norm_lpips +
        weights['clip'] * norm_clip
    )
    
    return combined_score

def load_image(image_path):
    """Load and convert image to numpy array."""
    return cv2.imread(image_path)

def calculate_psnr(img1, img2):
    """Calculate PSNR between two images."""
    return psnr(img1, img2)

def calculate_ssim(img1, img2):
    """Calculate SSIM between two images."""
    return ssim(img1, img2, channel_axis=2, data_range=255)

def calculate_lpips(img1_path, img2_path, lpips_model, device):
    """Calculate LPIPS between two images."""
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])
    
    # Load and transform images
    img1 = transform(Image.open(img1_path)).unsqueeze(0)
    img2 = transform(Image.open(img2_path)).unsqueeze(0)
    
   
    img1 = img1.to(device)
    img2 = img2.to(device)
    
    with torch.no_grad():
        lpips_score = lpips_model(img1, img2).item()
    return lpips_score

def calculate_clip_similarity(img1_path, img2_path, clip_model, clip_processor, device):
    """Calculate CLIP embedding cosine similarity between two images."""
    img1 = Image.open(img1_path)
    img2 = Image.open(img2_path)
    
    inputs1 = clip_processor(images=img1, return_tensors="pt")
    inputs2 = clip_processor(images=img2, return_tensors="pt")
    
    # Move inputs to device
    inputs1 = {k: v.to(device) if hasattr(v, 'to') else v for k, v in inputs1.items()}
    inputs2 = {k: v.to(device) if hasattr(v, 'to') else v for k, v in inputs2.items()}
    
    with torch.no_grad():
        img1_features = clip_model.get_image_features(**inputs1)
        img2_features = clip_model.get_image_features(**inputs2)
    
    img1_features = img1_features / img1_features.norm(dim=-1, keepdim=True)
    img2_features = img2_features / img2_features.norm(dim=-1, keepdim=True)
    
    similarity = (img1_features @ img2_features.T).item()
    return similarity

def main():
    # Set paths
    fake_dir = "/shared/shashmi/inversion/inversion/SD3.5/real"
    inverted_dir = "/shared/shashmi/inversion/inversion/SD3.5/inv_real"
    output_file = "SD3.5_uncondtioned_image_real_inv_real_metrics.jsonl"  # Output JSONL file path
    
    # Set device
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    
    # Initialize models
    print("Initializing models...")
    lpips_model = lpips.LPIPS(net='alex').to(device).eval()
    clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
    clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    
    # Get list of image files
    image_files = [f for f in os.listdir(fake_dir) if f.endswith('.png')]
    total_images = len(image_files)
    print(f"\nProcessing {total_images} images...")
    
    # Initialize dictionary to store results
    results = []
    
    # Open JSONL file for writing
    with open(output_file, 'w') as f:
        # Calculate metrics for each pair of images with progress bar
        for img_file in tqdm(image_files, desc="Processing images", unit="img"):
            fake_path = os.path.join(fake_dir, img_file)
            inverted_path = os.path.join(inverted_dir, img_file)
            
            # Load images for PSNR and SSIM
            fake_img = load_image(fake_path)
            inverted_img = load_image(inverted_path)
            
            # Calculate metrics
            psnr_value = calculate_psnr(fake_img, inverted_img)
            ssim_value = calculate_ssim(fake_img, inverted_img)
            lpips_value = calculate_lpips(fake_path, inverted_path, lpips_model, device)
            clip_sim = calculate_clip_similarity(fake_path, inverted_path, clip_model, clip_processor, device)
            
            # Calculate combined score
            combined_score = calculate_combined_score(psnr_value, ssim_value, lpips_value, clip_sim)
            
            # Create result dictionary
            result = {
                'image_path': fake_path,
                'metrics': {
                    'psnr': float(psnr_value),
                    'ssim': float(ssim_value),
                    'lpips': float(lpips_value),
                    'clip_similarity': float(clip_sim),
                    'combined_score': float(combined_score)
                }
            }
            
            # Write to JSONL file
            json.dump(result, f)
            f.write('\n')
            
            # Store results for printing
            results.append({
                'image': img_file,
                'psnr': psnr_value,
                'ssim': ssim_value,
                'lpips': lpips_value,
                'clip_similarity': clip_sim,
                'combined_score': combined_score
            })
    
    print("\nCalculating average metrics...")
    
    # Print results
    print("\nResults for each image pair:")
    print("-" * 80)
    for result in results:
        print(f"\nImage: {result['image']}")
        print(f"PSNR: {result['psnr']:.4f}")
        print(f"SSIM: {result['ssim']:.4f}")
        print(f"LPIPS: {result['lpips']:.4f}")
        print(f"CLIP Similarity: {result['clip_similarity']:.4f}")
        print(f"Combined Score: {result['combined_score']:.4f}")
    
    # Calculate and print average metrics
    avg_psnr = sum(r['psnr'] for r in results) / len(results)
    avg_ssim = sum(r['ssim'] for r in results) / len(results)
    avg_lpips = sum(r['lpips'] for r in results) / len(results)
    avg_clip = sum(r['clip_similarity'] for r in results) / len(results)
    avg_combined = sum(r['combined_score'] for r in results) / len(results)
    
    print("\nAverage Metrics:")
    print("-" * 80)
    print(f"Average PSNR: {avg_psnr:.4f}")
    print(f"Average SSIM: {avg_ssim:.4f}")
    print(f"Average LPIPS: {avg_lpips:.4f}")
    print(f"Average CLIP Similarity: {avg_clip:.4f}")
    print(f"Average Combined Score: {avg_combined:.4f}")
    
    print(f"\nResults have been saved to {output_file}")

if __name__ == "__main__":
    main()

/shared/shashmi/conda_envs/inverter/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Initializing models...
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/shared/shashmi/conda_envs/inverter/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/shared/shashmi/conda_envs/inverter/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /shared/shashmi/conda_envs/inverter/lib/python3.10/site-packages/lpips/weights/v0.1/alex.pth


/shared/shashmi/conda_envs/inverter/lib/python3.10/site-packages/lpips/lpips.py:107: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.load_state_dict(torch.load(model_path


Processing 1271 images...


Processing images:   0%|          | 0/1271 [00:00<?, ?img/s]


ValueError: Input images must have the same dimensions.

resized one

In [1]:
import os
import torch
import numpy as np
from PIL import Image
import torchvision.transforms as transforms
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import lpips
from transformers import CLIPProcessor, CLIPModel
import cv2
import json
from tqdm import tqdm

def normalize_psnr(psnr_value):
    """Normalize PSNR to 0-1 range"""
    min_psnr = 20
    max_psnr = 50
    return np.clip((psnr_value - min_psnr) / (max_psnr - min_psnr), 0, 1)

def calculate_combined_score(psnr_value, ssim_value, lpips_value, clip_sim):
    """Calculate combined similarity score with weighted metrics"""
    norm_psnr = normalize_psnr(psnr_value)
    norm_ssim = (ssim_value + 1) / 2
    norm_lpips = 1 - lpips_value
    norm_clip = (clip_sim + 1) / 2
    
    weights = {
        'psnr': 0.15,
        'ssim': 0.30,
        'lpips': 0.35,
        'clip': 0.20
    }
    
    combined_score = (
        weights['psnr'] * norm_psnr +
        weights['ssim'] * norm_ssim +
        weights['lpips'] * norm_lpips +
        weights['clip'] * norm_clip
    )
    
    return combined_score

def load_and_resize_image(image_path, target_size=None):
    """Load and convert image to numpy array, optionally resizing it."""
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Failed to load image: {image_path}")
    
    if target_size is not None:
        img = cv2.resize(img, target_size, interpolation=cv2.INTER_AREA)
    
    return img

def get_image_dimensions(image_path):
    """Get dimensions of an image."""
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Failed to load image: {image_path}")
    return img.shape[1], img.shape[0]  # width, height

def calculate_psnr(img1, img2):
    """Calculate PSNR between two images."""
    if img1.shape != img2.shape:
        raise ValueError("Images must have the same dimensions for PSNR calculation")
    return psnr(img1, img2)

def calculate_ssim(img1, img2):
    """Calculate SSIM between two images."""
    if img1.shape != img2.shape:
        raise ValueError("Images must have the same dimensions for SSIM calculation")
    return ssim(img1, img2, channel_axis=2, data_range=255)

def calculate_lpips(img1_path, img2_path, lpips_model, device):
    """Calculate LPIPS between two images."""
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])
    
    try:
        img1 = transform(Image.open(img1_path)).unsqueeze(0)
        img2 = transform(Image.open(img2_path)).unsqueeze(0)
    except Exception as e:
        raise ValueError(f"Failed to process images for LPIPS: {str(e)}")
    
    img1 = img1.to(device)
    img2 = img2.to(device)
    
    with torch.no_grad():
        lpips_score = lpips_model(img1, img2).item()
    return lpips_score

def calculate_clip_similarity(img1_path, img2_path, clip_model, clip_processor, device):
    """Calculate CLIP embedding cosine similarity between two images."""
    try:
        img1 = Image.open(img1_path)
        img2 = Image.open(img2_path)
    except Exception as e:
        raise ValueError(f"Failed to load images for CLIP similarity: {str(e)}")
    
    inputs1 = clip_processor(images=img1, return_tensors="pt")
    inputs2 = clip_processor(images=img2, return_tensors="pt")
    
    inputs1 = {k: v.to(device) if hasattr(v, 'to') else v for k, v in inputs1.items()}
    inputs2 = {k: v.to(device) if hasattr(v, 'to') else v for k, v in inputs2.items()}
    
    with torch.no_grad():
        img1_features = clip_model.get_image_features(**inputs1)
        img2_features = clip_model.get_image_features(**inputs2)
    
    img1_features = img1_features / img1_features.norm(dim=-1, keepdim=True)
    img2_features = img2_features / img2_features.norm(dim=-1, keepdim=True)
    
    similarity = (img1_features @ img2_features.T).item()
    return similarity

def main():
    # Set paths
    fake_dir = "/shared/shashmi/inversion/inversion/SD3.5/real"
    inverted_dir = "/shared/shashmi/inversion/inversion/SD3.5/inv_real"
    output_file = "SD3.5_uncondtioned_image_real_inv_real_metrics.jsonl"
    error_log = "processing_errors.log"
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    
    print("Initializing models...")
    lpips_model = lpips.LPIPS(net='alex').to(device).eval()
    clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
    clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    
    image_files = [f for f in os.listdir(fake_dir) if f.endswith('.png')]
    total_images = len(image_files)
    print(f"\nProcessing {total_images} images...")
    
    results = []
    errors = []
    
    with open(output_file, 'w') as f, open(error_log, 'w') as error_f:
        for img_file in tqdm(image_files, desc="Processing images", unit="img"):
            try:
                fake_path = os.path.join(fake_dir, img_file)
                inverted_path = os.path.join(inverted_dir, img_file)
                
                # Get dimensions of both images
                fake_dims = get_image_dimensions(fake_path)
                inverted_dims = get_image_dimensions(inverted_path)
                
                # Use the dimensions of the fake image as target size
                target_size = fake_dims
                
                # Load and resize images if necessary
                fake_img = load_and_resize_image(fake_path)
                inverted_img = load_and_resize_image(inverted_path, target_size)
                
                # Calculate metrics
                psnr_value = calculate_psnr(fake_img, inverted_img)
                ssim_value = calculate_ssim(fake_img, inverted_img)
                lpips_value = calculate_lpips(fake_path, inverted_path, lpips_model, device)
                clip_sim = calculate_clip_similarity(fake_path, inverted_path, clip_model, clip_processor, device)
                
                combined_score = calculate_combined_score(psnr_value, ssim_value, lpips_value, clip_sim)
                
                result = {
                    'image_path': fake_path,
                    'original_dimensions': {
                        'fake': fake_dims,
                        'inverted': inverted_dims
                    },
                    'metrics': {
                        'psnr': float(psnr_value),
                        'ssim': float(ssim_value),
                        'lpips': float(lpips_value),
                        'clip_similarity': float(clip_sim),
                        'combined_score': float(combined_score)
                    }
                }
                
                json.dump(result, f)
                f.write('\n')
                
                results.append({
                    'image': img_file,
                    'psnr': psnr_value,
                    'ssim': ssim_value,
                    'lpips': lpips_value,
                    'clip_similarity': clip_sim,
                    'combined_score': combined_score
                })
                
            except Exception as e:
                error_msg = f"Error processing {img_file}: {str(e)}"
                print(f"\nWarning: {error_msg}")
                errors.append(error_msg)
                error_f.write(f"{error_msg}\n")
                continue
    
    if results:
        print("\nCalculating average metrics...")
        
        print("\nResults for each image pair:")
        print("-" * 80)
        for result in results:
            print(f"\nImage: {result['image']}")
            print(f"PSNR: {result['psnr']:.4f}")
            print(f"SSIM: {result['ssim']:.4f}")
            print(f"LPIPS: {result['lpips']:.4f}")
            print(f"CLIP Similarity: {result['clip_similarity']:.4f}")
            print(f"Combined Score: {result['combined_score']:.4f}")
        
        avg_psnr = sum(r['psnr'] for r in results) / len(results)
        avg_ssim = sum(r['ssim'] for r in results) / len(results)
        avg_lpips = sum(r['lpips'] for r in results) / len(results)
        avg_clip = sum(r['clip_similarity'] for r in results) / len(results)
        avg_combined = sum(r['combined_score'] for r in results) / len(results)
        
        print("\nAverage Metrics:")
        print("-" * 80)
        print(f"Average PSNR: {avg_psnr:.4f}")
        print(f"Average SSIM: {avg_ssim:.4f}")
        print(f"Average LPIPS: {avg_lpips:.4f}")
        print(f"Average CLIP Similarity: {avg_clip:.4f}")
        print(f"Average Combined Score: {avg_combined:.4f}")
    
    if errors:
        print(f"\nEncountered {len(errors)} errors. See {error_log} for details.")
    
    print(f"\nResults have been saved to {output_file}")

if __name__ == "__main__":
    main()

/shared/shashmi/conda_envs/inverter/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Initializing models...
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/shared/shashmi/conda_envs/inverter/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/shared/shashmi/conda_envs/inverter/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /shared/shashmi/conda_envs/inverter/lib/python3.10/site-packages/lpips/weights/v0.1/alex.pth


/shared/shashmi/conda_envs/inverter/lib/python3.10/site-packages/lpips/lpips.py:107: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.load_state_dict(torch.load(model_path


Processing 1271 images...


Processing images:   0%|          | 0/1271 [00:00<?, ?img/s]libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
Processing images:   0%|          | 4/1271 [00:02<10:40,  1.98img/s]libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
Processing images:   1%|          | 12/1271 [00:04<06:17,  3.33img/s]libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
Processing images:   1%|▏         | 17/1271 [00:06<06:06,  3.42img/s]libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
Processing images:   2%|▏         | 25/1271 [00:08<06:34,  3.16img/s]libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
Processing images:   2%|▏         | 28/1271 [00:09<07:10,  2.89img/s]libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorre


Calculating average metrics...

Results for each image pair:
--------------------------------------------------------------------------------

Image: Motorcyclists_are_gathered_around_their_bikes_at_a_94d555f426.png
PSNR: 18.9109
SSIM: 0.5805
LPIPS: 0.1632
CLIP Similarity: 0.7300
Combined Score: 0.7030

Image: A_city_street_with_a_storm_drain__fire_hydrant__an_cbb718d675.png
PSNR: 19.3727
SSIM: 0.4191
LPIPS: 0.2741
CLIP Similarity: 0.7994
Combined Score: 0.6469

Image: A_green_and_yellow_train_sitting_under_a_metal_str_73d62c4f91.png
PSNR: 17.1658
SSIM: 0.4751
LPIPS: 0.1889
CLIP Similarity: 0.8989
Combined Score: 0.6950

Image: A_close_up_of_a_toilet_with_a_pink_stuffed_bunny_i_48279e5417.png
PSNR: 13.0168
SSIM: 0.4809
LPIPS: 0.3517
CLIP Similarity: 0.8321
Combined Score: 0.6323

Image: A_street_covered_in_pink_confetti_next_to_a_red_fi_5b7dd5e1d0.png
PSNR: 15.0120
SSIM: 0.1424
LPIPS: 0.5274
CLIP Similarity: 0.7535
Combined Score: 0.5121

Image: Light_from_an_open_door_shines_on_a_sim

In [2]:
import os
import torch
import numpy as np
from PIL import Image
import torchvision.transforms as transforms
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import lpips
from transformers import CLIPProcessor, CLIPModel
import cv2
import json
from tqdm import tqdm

def normalize_psnr(psnr_value):
    """Normalize PSNR to 0-1 range"""
    min_psnr = 20
    max_psnr = 50
    return np.clip((psnr_value - min_psnr) / (max_psnr - min_psnr), 0, 1)

def calculate_combined_score(psnr_value, ssim_value, lpips_value, clip_sim):
    """Calculate combined similarity score with weighted metrics"""
    norm_psnr = normalize_psnr(psnr_value)
    norm_ssim = (ssim_value + 1) / 2
    norm_lpips = 1 - lpips_value
    norm_clip = (clip_sim + 1) / 2
    
    weights = {
        'psnr': 0.15,
        'ssim': 0.30,
        'lpips': 0.35,
        'clip': 0.20
    }
    
    combined_score = (
        weights['psnr'] * norm_psnr +
        weights['ssim'] * norm_ssim +
        weights['lpips'] * norm_lpips +
        weights['clip'] * norm_clip
    )
    
    return combined_score

def load_and_resize_image(image_path, target_size=None):
    """Load and convert image to numpy array, optionally resizing it."""
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Failed to load image: {image_path}")
    
    if target_size is not None:
        img = cv2.resize(img, target_size, interpolation=cv2.INTER_AREA)
    
    return img

def get_image_dimensions(image_path):
    """Get dimensions of an image."""
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Failed to load image: {image_path}")
    return img.shape[1], img.shape[0]  # width, height

def calculate_psnr(img1, img2):
    """Calculate PSNR between two images."""
    if img1.shape != img2.shape:
        raise ValueError("Images must have the same dimensions for PSNR calculation")
    return psnr(img1, img2)

def calculate_ssim(img1, img2):
    """Calculate SSIM between two images."""
    if img1.shape != img2.shape:
        raise ValueError("Images must have the same dimensions for SSIM calculation")
    return ssim(img1, img2, channel_axis=2, data_range=255)

def calculate_lpips(img1_path, img2_path, lpips_model, device):
    """Calculate LPIPS between two images."""
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])
    
    try:
        img1 = transform(Image.open(img1_path)).unsqueeze(0)
        img2 = transform(Image.open(img2_path)).unsqueeze(0)
    except Exception as e:
        raise ValueError(f"Failed to process images for LPIPS: {str(e)}")
    
    img1 = img1.to(device)
    img2 = img2.to(device)
    
    with torch.no_grad():
        lpips_score = lpips_model(img1, img2).item()
    return lpips_score

def calculate_clip_similarity(img1_path, img2_path, clip_model, clip_processor, device):
    """Calculate CLIP embedding cosine similarity between two images."""
    try:
        img1 = Image.open(img1_path)
        img2 = Image.open(img2_path)
    except Exception as e:
        raise ValueError(f"Failed to load images for CLIP similarity: {str(e)}")
    
    inputs1 = clip_processor(images=img1, return_tensors="pt")
    inputs2 = clip_processor(images=img2, return_tensors="pt")
    
    inputs1 = {k: v.to(device) if hasattr(v, 'to') else v for k, v in inputs1.items()}
    inputs2 = {k: v.to(device) if hasattr(v, 'to') else v for k, v in inputs2.items()}
    
    with torch.no_grad():
        img1_features = clip_model.get_image_features(**inputs1)
        img2_features = clip_model.get_image_features(**inputs2)
    
    img1_features = img1_features / img1_features.norm(dim=-1, keepdim=True)
    img2_features = img2_features / img2_features.norm(dim=-1, keepdim=True)
    
    similarity = (img1_features @ img2_features.T).item()
    return similarity

def main():
    # Set paths
    fake_dir = "/shared/shashmi/inversion/inversion/SD3/real"
    inverted_dir = "/shared/shashmi/inversion/inversion/SD3/inv_real"
    output_file = "SD3_uncondtioned_image_real_inv_real_metrics.jsonl"
    error_log = "processing_errors.log"
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    
    print("Initializing models...")
    lpips_model = lpips.LPIPS(net='alex').to(device).eval()
    clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
    clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    
    image_files = [f for f in os.listdir(fake_dir) if f.endswith('.png')]
    total_images = len(image_files)
    print(f"\nProcessing {total_images} images...")
    
    results = []
    errors = []
    
    with open(output_file, 'w') as f, open(error_log, 'w') as error_f:
        for img_file in tqdm(image_files, desc="Processing images", unit="img"):
            try:
                fake_path = os.path.join(fake_dir, img_file)
                inverted_path = os.path.join(inverted_dir, img_file)
                
                # Get dimensions of both images
                fake_dims = get_image_dimensions(fake_path)
                inverted_dims = get_image_dimensions(inverted_path)
                
                # Use the dimensions of the fake image as target size
                target_size = fake_dims
                
                # Load and resize images if necessary
                fake_img = load_and_resize_image(fake_path)
                inverted_img = load_and_resize_image(inverted_path, target_size)
                
                # Calculate metrics
                psnr_value = calculate_psnr(fake_img, inverted_img)
                ssim_value = calculate_ssim(fake_img, inverted_img)
                lpips_value = calculate_lpips(fake_path, inverted_path, lpips_model, device)
                clip_sim = calculate_clip_similarity(fake_path, inverted_path, clip_model, clip_processor, device)
                
                combined_score = calculate_combined_score(psnr_value, ssim_value, lpips_value, clip_sim)
                
                result = {
                    'image_path': fake_path,
                    'original_dimensions': {
                        'fake': fake_dims,
                        'inverted': inverted_dims
                    },
                    'metrics': {
                        'psnr': float(psnr_value),
                        'ssim': float(ssim_value),
                        'lpips': float(lpips_value),
                        'clip_similarity': float(clip_sim),
                        'combined_score': float(combined_score)
                    }
                }
                
                json.dump(result, f)
                f.write('\n')
                
                results.append({
                    'image': img_file,
                    'psnr': psnr_value,
                    'ssim': ssim_value,
                    'lpips': lpips_value,
                    'clip_similarity': clip_sim,
                    'combined_score': combined_score
                })
                
            except Exception as e:
                error_msg = f"Error processing {img_file}: {str(e)}"
                print(f"\nWarning: {error_msg}")
                errors.append(error_msg)
                error_f.write(f"{error_msg}\n")
                continue
    
    if results:
        print("\nCalculating average metrics...")
        
        print("\nResults for each image pair:")
        print("-" * 80)
        for result in results:
            print(f"\nImage: {result['image']}")
            print(f"PSNR: {result['psnr']:.4f}")
            print(f"SSIM: {result['ssim']:.4f}")
            print(f"LPIPS: {result['lpips']:.4f}")
            print(f"CLIP Similarity: {result['clip_similarity']:.4f}")
            print(f"Combined Score: {result['combined_score']:.4f}")
        
        avg_psnr = sum(r['psnr'] for r in results) / len(results)
        avg_ssim = sum(r['ssim'] for r in results) / len(results)
        avg_lpips = sum(r['lpips'] for r in results) / len(results)
        avg_clip = sum(r['clip_similarity'] for r in results) / len(results)
        avg_combined = sum(r['combined_score'] for r in results) / len(results)
        
        print("\nAverage Metrics:")
        print("-" * 80)
        print(f"Average PSNR: {avg_psnr:.4f}")
        print(f"Average SSIM: {avg_ssim:.4f}")
        print(f"Average LPIPS: {avg_lpips:.4f}")
        print(f"Average CLIP Similarity: {avg_clip:.4f}")
        print(f"Average Combined Score: {avg_combined:.4f}")
    
    if errors:
        print(f"\nEncountered {len(errors)} errors. See {error_log} for details.")
    
    print(f"\nResults have been saved to {output_file}")

if __name__ == "__main__":
    main()

Using device: cuda
Initializing models...
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /shared/shashmi/conda_envs/inverter/lib/python3.10/site-packages/lpips/weights/v0.1/alex.pth

Processing 7000 images...


Processing images:   0%|          | 0/7000 [00:00<?, ?img/s]libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
Processing images:   0%|          | 1/7000 [00:00<37:37,  3.10img/s]libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
Processing images:   0%|          | 2/7000 [00:00<33:54,  3.44img/s]libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
Processing images:   0%|          | 11/7000 [00:03<33:43,  3.45img/s]libpng warning: iCCP: profile 'ICC Profile': 1000000h: invalid rendering intent
libpng warning: iCCP: profile 'ICC Profile': 1000000h: invalid rendering intent
Processing images:   0%|          | 16/7000 [00:04<33:40,  3.46img/s]libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
Processing images:   0%|          | 28/7000 [00:08<33:31,  3.47img/s]libpng warning: iCCP: known i


Calculating average metrics...

Results for each image pair:
--------------------------------------------------------------------------------

Image: A_motorcycle_parked_in_a_parking_space_next_to_ano_8714764ce6.png
PSNR: 12.5091
SSIM: 0.1005
LPIPS: 0.2043
CLIP Similarity: 0.9093
Combined Score: 0.6345

Image: Motorcyclists_are_gathered_around_their_bikes_at_a_94d555f426.png
PSNR: 15.1458
SSIM: 0.4713
LPIPS: 0.2690
CLIP Similarity: 0.8293
Combined Score: 0.6595

Image: A_toy_train_track_is_set_up_with_two_trains__house_91e94bb534.png
PSNR: 16.0157
SSIM: 0.4228
LPIPS: 0.4256
CLIP Similarity: 0.6565
Combined Score: 0.5801

Image: A_group_of_cows_on_street_next_to_building_and_bus_35d59390c5.png
PSNR: 12.9426
SSIM: 0.3180
LPIPS: 0.2971
CLIP Similarity: 0.8755
Combined Score: 0.6313

Image: Sign_on_the_side_of_the_road_welcoming_people_to_A_7b96469d50.png
PSNR: 13.4267
SSIM: 0.4519
LPIPS: 0.2121
CLIP Similarity: 0.9797
Combined Score: 0.6915

Image: A_couple_of_dogs_standing_and_laying_on